# Cryptocurrency Data Preprocessing
This notebook loads all raw cryptocurrency CSV files from `data/raw/`,
cleans and preprocesses the data, engineers key features, and saves
the final combined dataset to `data/processed/combined_crypto_data.csv`.

In [ ]:
import pandas as pd
import numpy as np
import os

## Load All CSVs & Combine
Load each cryptocurrency CSV from `data/raw/`, apply basic cleaning steps,
add a `Crypto` label column, and combine all into a single DataFrame.

**Steps per file:**
- Strip extra spaces from column names
- Parse Date column to datetime
- Sort by date chronologically
- Convert OHLCV columns to numeric
- Drop `Adj Close` (redundant for crypto)
- Add `Crypto` identifier column

In [ ]:
folder_path = "../data/raw"
all_data = []

for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        crypto_name = file.replace(".csv", "")
        file_path = os.path.join(folder_path, file)
        print(f"Loading {crypto_name}...")

        df = pd.read_csv(file_path)

        # Strip extra spaces from column names
        df.columns = df.columns.str.strip()

        # Parse date
        df["Date"] = pd.to_datetime(df["Date"])

        # Sort by date
        df = df.sort_values("Date").reset_index(drop=True)

        # Convert numeric columns
        numeric_cols = ["Open", "High", "Low", "Close", "Volume"]
        for col in numeric_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        # Drop Adj Close if present
        if "Adj Close" in df.columns:
            df = df.drop(columns=["Adj Close"])

        # Add crypto label
        df["Crypto"] = crypto_name

        all_data.append(df)

# Combine all
combined_df = pd.concat(all_data, ignore_index=True)
combined_df = combined_df.sort_values(["Date", "Crypto"]).reset_index(drop=True)

print(f"\nLoaded {len(all_data)} cryptos | Combined shape: {combined_df.shape}")
combined_df.head(10)

Loading ADA...
Loading BNB...
Loading BTC...
Loading DOGE...
Loading DOT...
Loading ETH...
Loading LINK...
Loading LTC...
Loading SOL...
Loading XRP...

Loaded 10 cryptos | Combined shape: (18250, 7)


,Date,Open,High,Low,Close,Volume,Crypto
0,2021-01-01,0.181382,0.184246,0.172022,0.175350,1122218004,ADA
1,2021-01-01,37.374573,38.928177,37.046307,37.905010,459165743,BNB
2,2021-01-01,28994.009766,29600.626953,28803.585938,29374.152344,40730301359,BTC
3,2021-01-01,0.004681,0.005685,0.004615,0.005685,228961515,DOGE
4,2021-01-01,9.288180,9.416587,8.120194,8.306819,2791211123,DOT
5,2021-01-01,737.708374,749.201843,719.792236,730.367554,13652004358,ETH
6,2021-01-01,11.266762,12.386067,11.118653,11.872555,1376173841,LINK
7,2021-01-01,124.672768,133.185760,123.328079,126.230347,7326980728,LTC
8,2021-01-01,1.509775,1.859656,1.502038,1.842084,25722549,SOL
9,2021-01-01,0.219845,0.249270,0.217288,0.237444,5888429287,XRP


## Basic Dataset Info
Verify the date range, coins included, and row counts per cryptocurrency.

In [ ]:
print("Date Range:")
print("  From :", combined_df["Date"].min().date())
print("  To   :", combined_df["Date"].max().date())

print("\nCoins Included:")
print(combined_df["Crypto"].unique())

print("\nRows per Crypto:")
print(combined_df["Crypto"].value_counts())

Date Range:
  From : 2021-01-01
  To   : 2025-12-30

Coins Included:
['ADA' 'BNB' 'BTC' 'DOGE' 'DOT' 'ETH' 'LINK' 'LTC' 'SOL' 'XRP']

Rows per Crypto:
Crypto
ADA     1825
BNB     1825
BTC     1825
DOGE    1825
DOT     1825
ETH     1825
LINK    1825
LTC     1825
SOL     1825
XRP     1825
Name: count, dtype: int64


## Handle Missing Values
Check for missing values in each column and handle them using
**forward fill followed by backward fill** per cryptocurrency group.

In [ ]:
print("Missing Values Before:")
print(combined_df.isnull().sum())

numeric_cols = ["Open", "High", "Low", "Close", "Volume"]

# Forward fill & backward fill per crypto
combined_df = combined_df.sort_values(["Crypto", "Date"])
combined_df[numeric_cols] = combined_df.groupby("Crypto")[numeric_cols].transform(
    lambda x: x.ffill().bfill()
)

print("\nMissing Values After:")
print(combined_df.isnull().sum())

Missing Values Before:
Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
Crypto    0
dtype: int64

Missing Values After:
Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
Crypto    0
dtype: int64


## Feature Engineering
Add derived features to enrich the dataset for downstream analysis and modeling.

| Feature | Formula | Purpose |
|---------|---------|---------|
| Daily_Return | (Close_t / Close_t-1 - 1) × 100 | % price change per day |
| Log_Return | log(Close_t / Close_t-1) | Normalized return for modeling |
| Price_Range | High - Low | Intraday volatility indicator |
| MA_7 | 7-day rolling mean of Close | Short-term trend |
| MA_30 | 30-day rolling mean of Close | Medium-term trend |
| Volatility_7 | 7-day rolling std of Daily_Return | Short-term risk measure |
| Cumulative_Return | (Close_t / Close_0 - 1) × 100 | Total return from start date |

In [ ]:
combined_df = combined_df.sort_values(["Crypto", "Date"]).reset_index(drop=True)

# Daily return (%)
combined_df["Daily_Return"] = combined_df.groupby("Crypto")["Close"].pct_change() * 100

# Log return
combined_df["Log_Return"] = combined_df.groupby("Crypto")["Close"].transform(
    lambda x: np.log(x / x.shift(1))
)

# Price range
combined_df["Price_Range"] = combined_df["High"] - combined_df["Low"]

# Moving averages
combined_df["MA_7"]  = combined_df.groupby("Crypto")["Close"].transform(lambda x: x.rolling(7).mean())
combined_df["MA_30"] = combined_df.groupby("Crypto")["Close"].transform(lambda x: x.rolling(30).mean())

# Rolling volatility (7-day std of daily returns)
combined_df["Volatility_7"] = combined_df.groupby("Crypto")["Daily_Return"].transform(
    lambda x: x.rolling(7).std()
)

# Cumulative return from start
combined_df["Cumulative_Return"] = combined_df.groupby("Crypto")["Close"].transform(
    lambda x: (x / x.iloc[0] - 1) * 100
)

print("Features Added:", combined_df.columns.tolist())
combined_df.head()

Features Added: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Crypto', 'Daily_Return', 'Log_Return', 'Price_Range', 'MA_7', 'MA_30', 'Volatility_7', 'Cumulative_Return']


,Date,Open,High,Low,Close,Volume,Crypto,Daily_Return,Log_Return,Price_Range,MA_7,MA_30,Volatility_7,Cumulative_Return
0,2021-01-01,0.181382,0.184246,0.172022,0.175350,1122218004,ADA,NaN,NaN,0.012224,NaN,NaN,NaN,0.000000
1,2021-01-02,0.175359,0.184253,0.169233,0.177423,1408849504,ADA,1.182210,0.011753,0.015020,NaN,NaN,NaN,1.182210
2,2021-01-03,0.177382,0.208679,0.173376,0.204995,2303857909,ADA,15.540266,0.144449,0.035303,NaN,NaN,NaN,16.906194
3,2021-01-04,0.205236,0.239661,0.194450,0.224762,3260699086,ADA,9.642667,0.092056,0.045211,NaN,NaN,NaN,28.179070
4,2021-01-05,0.224817,0.264886,0.208454,0.258314,4097207384,ADA,14.927800,0.139134,0.056432,NaN,NaN,NaN,47.313385


## Final Dataset Check
Verify the final shape, data types, and descriptive statistics
per cryptocurrency before saving.

In [ ]:
print("Final Shape:", combined_df.shape)

print("\nData Types:")
print(combined_df.dtypes)

print("\nStats per Crypto:")
combined_df.groupby("Crypto")["Close"].describe().reset_index()

Final Shape: (18250, 14)

Data Types:
Date                 datetime64[ns]
Open                        float64
High                        float64
Low                         float64
Close                       float64
Volume                        int64
Crypto                       object
Daily_Return                float64
Log_Return                  float64
Price_Range                 float64
MA_7                        float64
MA_30                       float64
Volatility_7                float64
Cumulative_Return           float64
dtype: object

Stats per Crypto:


,Crypto,count,mean,std,min,25%,50%,75%,max
0,ADA,1825.0,0.746070,0.517767,0.175350,0.376906,0.565697,0.932902,2.968239
1,BNB,1825.0,456.744587,220.630286,37.905010,286.991058,400.422394,599.681702,1310.214355
2,BTC,1825.0,54408.228931,29461.698618,15787.284180,29397.714844,46453.566406,69122.335938,124752.531250
3,DOGE,1825.0,0.150136,0.094895,0.005685,0.075783,0.128088,0.198864,0.684777
4,DOT,1825.0,11.333598,10.803588,1.682400,4.547215,6.447359,14.539050,53.881733
5,ETH,1825.0,2534.555871,911.742806,730.367554,1792.485107,2448.921143,3236.134277,4831.348633
6,LINK,1825.0,15.559485,8.051332,5.119156,7.673118,14.260993,20.001366,52.198696
7,LTC,1825.0,105.060120,49.485301,43.300301,70.924530,89.744026,120.269012,386.450775
8,SOL,1825.0,98.548326,71.560524,1.799275,27.376421,97.879982,156.998062,261.869751
9,XRP,1825.0,1.024320,0.822801,0.221655,0.485699,0.611899,1.216473,3.555765


## Save Processed Data
Save the final combined and feature-engineered dataset to `data/processed/`.
This file will serve as the single input source for all downstream notebooks.

In [ ]:
os.makedirs("../data/processed", exist_ok=True)
combined_df.to_csv("../data/processed/combined_crypto_data.csv", index=False)
print("Saved: ../data/processed/combined_crypto_data.csv")
print("Final Shape:", combined_df.shape)
combined_df.head()

Saved: ../data/processed/combined_crypto_data.csv
Final Shape: (18250, 14)


,Date,Open,High,Low,Close,Volume,Crypto,Daily_Return,Log_Return,Price_Range,MA_7,MA_30,Volatility_7,Cumulative_Return
0,2021-01-01,0.181382,0.184246,0.172022,0.175350,1122218004,ADA,NaN,NaN,0.012224,NaN,NaN,NaN,0.000000
1,2021-01-02,0.175359,0.184253,0.169233,0.177423,1408849504,ADA,1.182210,0.011753,0.015020,NaN,NaN,NaN,1.182210
2,2021-01-03,0.177382,0.208679,0.173376,0.204995,2303857909,ADA,15.540266,0.144449,0.035303,NaN,NaN,NaN,16.906194
3,2021-01-04,0.205236,0.239661,0.194450,0.224762,3260699086,ADA,9.642667,0.092056,0.045211,NaN,NaN,NaN,28.179070
4,2021-01-05,0.224817,0.264886,0.208454,0.258314,4097207384,ADA,14.927800,0.139134,0.056432,NaN,NaN,NaN,47.313385


## Summary

| Step | Action | Result |
|------|--------|--------|
| Load | Read 10 CSV files from data/raw/ | Combined DataFrame |
| Clean | Fix dtypes, strip spaces, sort dates | Consistent schema |
| Missing Values | Forward fill + backward fill per crypto | Zero missing values |
| Features | Added 7 engineered features | 14 total columns |
| Save | combined_crypto_data.csv | data/processed/ |

**Final Columns:**
`Date, Open, High, Low, Close, Volume, Crypto,
Daily_Return, Log_Return, Price_Range, MA_7, MA_30, Volatility_7, Cumulative_Return`
